# Ordinary kriging: estimate values and inspect uncertainty

Ordinary kriging uses a spatial dependence model to calculate prediction weights. This lab compares variogram choices and shows both the predicted surface and the kriging variance.

We will use a synthetic dataset so the example is reproducible. The controls refit the model when you change the variogram family or number of lag bins.

**You need:** NumPy, Matplotlib, PyKrige and ipywidgets.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import Dropdown, IntSlider, interact
from pykrige.ok import OrdinaryKriging

rng = np.random.default_rng(42)
x = rng.uniform(0, 10, 24)
y = rng.uniform(0, 10, 24)
z = np.sin(x / 2) + np.cos(y / 3) + rng.normal(0, 0.15, size=len(x))

grid_x = np.linspace(0, 10, 65)
grid_y = np.linspace(0, 10, 65)

## 1. Inspect the sample locations

The observations follow a smooth synthetic spatial pattern with added noise. This is a teaching dataset, not a measurement campaign.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
sc = ax.scatter(x, y, c=z, s=85, edgecolor="black")
fig.colorbar(sc, ax=ax, label="Observed value")
ax.set(xlabel="x coordinate", ylabel="y coordinate", title="Synthetic sample locations")
ax.set_aspect("equal")
plt.show()

## 2. Fit a variogram and predict a grid

The library estimates variogram parameters from the empirical semivariogram when none are supplied. We will change the model family and the number of lag bins, then inspect the prediction and the variance separately.

A different curve is not automatically a better model. Use validation and domain knowledge when analysing real data.

In [ ]:
def explore(variogram="spherical", lag_bins=6):
    model = OrdinaryKriging(
        x, y, z,
        variogram_model=variogram,
        nlags=lag_bins,
        verbose=False,
        enable_plotting=False,
    )
    prediction, variance = model.execute("grid", grid_x, grid_y)
    prediction = np.asarray(prediction)
    variance = np.asarray(variance)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
    image0 = axes[0].imshow(
        prediction, origin="lower", extent=(0, 10, 0, 10),
        aspect="equal", cmap="viridis"
    )
    axes[0].scatter(x, y, c="white", edgecolor="black", s=25)
    axes[0].set_title(f"Ordinary kriging prediction ({variogram})")
    axes[0].set_xlabel("x coordinate")
    axes[0].set_ylabel("y coordinate")
    fig.colorbar(image0, ax=axes[0], label="Predicted value")

    image1 = axes[1].imshow(
        variance, origin="lower", extent=(0, 10, 0, 10),
        aspect="equal", cmap="magma"
    )
    axes[1].scatter(x, y, c="white", edgecolor="black", s=25)
    axes[1].set_title("Kriging variance")
    axes[1].set_xlabel("x coordinate")
    axes[1].set_ylabel("y coordinate")
    fig.colorbar(image1, ax=axes[1], label="Model-based variance")
    plt.tight_layout()
    plt.show()

    print("Variogram model:", variogram)
    print("Lag bins:", lag_bins)
    print("Fitted parameters:", model.variogram_model_parameters)
    print("Variance is conditional on this fitted model and its assumptions.")

interact(
    explore,
    variogram=Dropdown(options=["spherical", "exponential", "gaussian"], value="spherical",
                       description="Variogram"),
    lag_bins=IntSlider(value=6, min=4, max=12, step=1, description="Lag bins"),
);

## 3. What to look for

- The prediction surface is influenced by both sample values and their spatial arrangement.
- The variance surface is influenced by the sampling geometry and fitted variogram.
- Areas with sparse support often have greater kriging variance, but do not assume this pattern without inspecting the result.
- Changing the variogram family changes the spatial dependence model and can change both outputs.

## 4. Questions to investigate

1. Where is the prediction variance highest?
2. How does the variance change near sampled locations?
3. Does switching from spherical to exponential substantially change the prediction?
4. What happens when the number of lag bins changes?
5. Why should you compare these results with IDW and spatially blocked validation?

## 5. Important limits

The notebook does not include a real-world dataset, trend modelling, directional anisotropy or uncertainty in the fitted variogram parameters. It demonstrates the ordinary-kriging workflow, not a complete geostatistical study.

Continue with the [ordinary kriging lesson](../algorithms/interpolation/ordinary-kriging.md), [empirical semivariogram](../methods/empirical-semivariogram.md) and [IDW lesson](../algorithms/interpolation/inverse-distance-weighting.md).